## 1. Session Overview

This notebook fine-tunes `meta-llama/Meta-Llama-3.1-8B-Instruct` using the MaxText framework on Kaggle TPU v5e.

- Objective: Run a minimal verification on TPU v5e, then proceed to fine-tuning.
- Evidence of Done: Successful `steps: 1` MaxText run and logs confirming TPU utilization.
- Artifacts: Config file, logs, and checkpoints saved to Kaggle outputs and/or GCS.

Preconditions:
- Kaggle accelerator set to TPU v5e.
- Internet enabled for cloning and dependency installs.
- Access to MaxText-compatible Llama 3.1 checkpoint via Kaggle Datasets.


## 2. Kaggle TPU v5e Environment Setup Plan

Steps in this session:
1. Verify TPU visibility and JAX version
2. Clone MaxText (main branch)
3. Install dependencies from `requirements.txt`
4. Prepare minimal `config.yaml` for verification run
5. Run a 1-step verification to confirm TPU v5e works

Notes:
- No substeps for now; each step maps to a single cell or small group of cells.
- We will capture logs and versions for reproducibility.


In [1]:
# 3. Verify TPU visibility and JAX environment
import os, sys, platform, subprocess, json

print("Python:", sys.version)
print("Platform:", platform.platform())

# Kaggle TPU env vars
for key in ["TPU_NAME", "TPU_WORKER_ID", "TPU_CHIPS_PER_PROCESS", "TPU_MULTISLICE_CTRL_ADDRESS"]:
    if key in os.environ:
        print(f"{key}:", os.environ[key])

try:
    import jax
    import jaxlib
    import jax.numpy as jnp
    print("jax:", jax.__version__)
    print("jaxlib:", jaxlib.__version__)
    devices = jax.devices()
    print("Devices:")
    for d in devices:
        print(" -", d)
    print("Device count:", len(devices))
    x = jnp.ones((8, 8))
    y = jnp.dot(x, x).block_until_ready()
    print("JAX test dot result shape:", y.shape)
except Exception as e:
    print("[ERROR] JAX/TPU verification failed:", e)
    raise


Python: 3.10.18 (main, Jul  1 2025, 05:26:40) [GCC 12.2.0]
Platform: Linux-6.1.42+-x86_64-with-glibc2.36
TPU_WORKER_ID: 0
jax: 0.4.34
jaxlib: 0.4.34


E0000 00:00:1758028206.802635      10 common_lib.cc:612] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:230


Devices:
 - TPU_0(process=0,(0,0,0,0))
 - TPU_1(process=0,(1,0,0,0))
 - TPU_2(process=0,(0,1,0,0))
 - TPU_3(process=0,(1,1,0,0))
 - TPU_4(process=0,(0,2,0,0))
 - TPU_5(process=0,(1,2,0,0))
 - TPU_6(process=0,(0,3,0,0))
 - TPU_7(process=0,(1,3,0,0))
Device count: 8
JAX test dot result shape: (8, 8)


## 4. Clone MaxText (main branch)

We will clone the official `google/maxtext` repository at the default `main` branch for the latest TPU v5e-compatible training scripts. Evidence of done: repository present in the working directory and HEAD commit printed.


In [2]:
%%bash
set -e

echo "Cloning google/maxtext (main)..."
if [ ! -d "maxtext" ]; then
  git clone --depth=1 https://github.com/google/maxtext.git
else
  echo "Repository 'maxtext' already exists; skipping clone."
fi

cd maxtext
echo "Repo HEAD:"
git log -1 --pretty=oneline || true

echo "Top-level files:"
ls -1 | sed -n '1,50p'


Cloning google/maxtext (main)...


Cloning into 'maxtext'...


Repo HEAD:
a55e18af31a76179e589314878af0a5195e7d7bd Merge pull request #2278 from AI-Hypercomputer:collabs-examples-sft
Top-level files:
AUTHORS
CONTRIBUTING.md
LICENSE
PREFLIGHT.md
README.md
RESTRUCTURE.md
benchmarks
clean_py_env.Dockerfile
code_style.sh
docker_build_dependency_image.sh
docker_upload_runner.sh
docs
download_dataset.sh
end_to_end
gpu_multi_process_run.sh
maxtext_custom_wheels.Dockerfile
maxtext_db_dependencies.Dockerfile
maxtext_dependencies.Dockerfile
maxtext_gpu_dependencies.Dockerfile
maxtext_jax_ai_image.Dockerfile
maxtext_libtpu_path.Dockerfile
maxtext_runner.Dockerfile
multihost_job.py
multihost_runner.py
pedagogical_examples
preflight.sh
pylintrc
pyproject.toml
pytest.ini
requirements.txt
requirements_docs.txt
requirements_with_jax_ai_image.txt
requirements_with_jax_stable_stack_0_6_1_pipreqs.txt
rto_setup.sh
setup.sh
setup_gcsfuse.sh
setup_with_retries.sh
src
tests
unit_test_and_lint.sh


## 5. Install dependencies from requirements.txt

Install Python dependencies required by MaxText. Kaggle TPU v5e includes a modern JAX stack; if a conflict arises, we will prefer the preinstalled JAX. Evidence of done: successful pip install and import checks.


In [3]:
%%bash
set -e

python -V
pip -V

echo "Installing MaxText requirements..."
pip install --no-input --no-cache-dir -r maxtext/requirements.txt

python - <<'PY'
import jax, jaxlib
print("jax:", jax.__version__)
print("jaxlib:", jaxlib.__version__)
print("JAX import successful after requirements install.")
PY


Python 3.10.18
pip 23.0.1 from /usr/local/lib/python3.10/site-packages/pip (python 3.10)
Installing MaxText requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━�0m━━━━━━━━━�━ 0.0/538.6 kB ? eta -:--:--��━━━━━━━━━━━━━━━━━ 41.0/538.6 kB 1.4 MB/s eta 0:00:01��━━━━━━━━━━━━━━━━━ 81.9/538.6 kB 1.5 MB/s eta 0:00:01╺━━━━━━━━━━━ 378.9/538.6 kB 3.8 MB/s eta 0:00:01��━━━━━━━━━━ 538.6/538.6 kB 4.3 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
     \ 2.8 MB 6.2 MB/s 0:00:000m
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
     - 417.8 kB 11.3 MB/s 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'do

  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [5 lines of output]
      Package sentencepiece was not found in the pkg-config search path.
      Perhaps you should add the directory containing `sentencepiece.pc'
      to the PKG_CONFIG_PATH environment variable
      Package 'sentencepiece', required by 'virtual:world', not found
      Failed to find sentencepiece pkgconfig
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.

[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


CalledProcessError: Command 'b'set -e\n\npython -V\npip -V\n\necho "Installing MaxText requirements..."\npip install --no-input --no-cache-dir -r maxtext/requirements.txt\n\npython - <<\'PY\'\nimport jax, jaxlib\nprint("jax:", jax.__version__)\nprint("jaxlib:", jaxlib.__version__)\nprint("JAX import successful after requirements install.")\nPY\n'' returned non-zero exit status 1.